# 02 · Experimento mínimo de punta a punta

**Etapa 2 del plan.** Corre el control **A** (lag y delta 1 y 2, la consigna) contra **una** variante histórica (por defecto **LT4**), con poco cómputo, para:

- validar el código: variables históricas, Optuna, semillas, ganancia y retomar una corrida cortada;
- medir tiempos y memoria reales, para presupuestar las corridas grandes;
- tener una primera idea de si hay señal.

**No** saca conclusiones sobre la hipótesis. **201909 y 201910 no se cargan**: quedan sellados hasta el pre-registro.

Lee `data/base_2019.parquet` y `resultados/01_meses_rotos.csv`, que salen del `01`. Ficha: `fichas/etapa_2_experimento_minimo.md` (en el laboratorio).

## 1. Parámetros

Las decisiones de la ficha están marcadas con su número. Las **provisorias** dependen de la respuesta de los profes: cambiarlas es cambiar un valor.

- **Corrida de prueba:** `N_PRUEBAS_OPTUNA = 2`, `N_SEMILLAS = 1` y `ARBOLES_MAX = 200` dan el orden de magnitud de los tiempos en pocos minutos.
- Cada combinación de `MESES_TRAIN`, `TRATAR_ROTOS` y `HISTORIA_SOBRE` es una **configuración** con su propia carpeta en `exp/02_minimo/` y su propio estudio de Optuna. Si la corrida se corta, se vuelve a correr entera: retoma donde quedó.

In [1]:
from pathlib import Path
import platform
import resource
import sys
import time
import warnings

import duckdb
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
RESULTADOS = Path('../resultados')
EXP = Path('../exp/02_minimo')

MESES_TRAIN = [201905, 201906]      # decisión 1 (provisorio)
MES_VALID = 201908                  # Optuna y ganancia de esta etapa (decisión 7, provisorio)
VARIANTE = 'LT4'                    # decisión 4
TRATAR_ROTOS = True                 # decisión 2 (provisorio): meses rotos a NA antes de la historia
HISTORIA_SOBRE = 'nominal'          # decisión 6: 'nominal' o 'ranking' (solo montos en pesos)

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 5                      # las primeras 5 de las 10 del TP
POSITIVOS = ['BAJA+1', 'BAJA+2']    # clase de entrenamiento (provisorio)
POSITIVOS_AUC = ['BAJA+2']          # la AUC de validación mira lo que paga

N_PRUEBAS_OPTUNA = 20               # decisión 8
OPTUNA_VARIANTE = False             # decisión 8 (b): búsqueda propia de la variante, solo para comparar
N_PRUEBAS_VARIANTE = 10
ARBOLES_MAX = 3000                  # tope del early stopping
PACIENCIA = 100
N_HILOS = 8                         # fijo: la VM (8 vCPU) y la Mac dan lo mismo

GANANCIA_ACIERTO = 1_072_500        # provisorio (pregunta 3)
COSTO_ESTIMULO = 27_500
VENTANA_SUAVIZADO = 501             # media móvil centrada, como z0607
LARGO_CURVA = 25_000

# Para la proyección del diseño completo (sección 13)
DISENO = {'variantes': 7, 'semillas': 10, 'meses_evaluacion': 2, 'pruebas_optuna': 50}
# -----------------------------------------------------------------------------

VARIANTES = {   # lags además de 1 y 2, y largo de la tendencia (ventana de m − h a m)
    'A':   {'lags': [],     'tendencia': None},
    'L3':  {'lags': [3],    'tendencia': None},
    'L4':  {'lags': [3, 4], 'tendencia': None},
    'T3':  {'lags': [],     'tendencia': 3},
    'T4':  {'lags': [],     'tendencia': 4},
    'LT3': {'lags': [3],    'tendencia': 3},
    'LT4': {'lags': [3, 4], 'tendencia': 4},
}

CONFIG = f"train{MESES_TRAIN[0]}-{MESES_TRAIN[-1]}_{'rotosNA' if TRATAR_ROTOS else 'rotos'}_{HISTORIA_SOBRE}"
EXP_CONFIG = EXP / CONFIG
CURVAS = EXP_CONFIG / 'curvas'
STORAGE = f"sqlite:///{(EXP_CONFIG / 'optuna.db').as_posix()}"
ROTOS_CSV = RESULTADOS / '01_meses_rotos.csv'
COBERTURA_CSV = RESULTADOS / '01_cobertura_historia.csv'
VARIABLES_CSV = RESULTADOS / '02_variables.csv'
SEMILLAS_CSV = RESULTADOS / '02_minimo_semillas.csv'
TIEMPOS_CSV = RESULTADOS / '02_tiempos.csv'

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14
optuna.logging.set_verbosity(optuna.logging.WARNING)
CURVAS.mkdir(parents=True, exist_ok=True)

assert BASE_PARQUET.exists(), f'Falta {BASE_PARQUET.resolve()}: correr antes el 01'
assert VARIANTE in VARIANTES and VARIANTE != 'A'
assert HISTORIA_SOBRE in ('nominal', 'ranking')
assert max(MESES_TRAIN) < MES_VALID < 201909, '201909 y 201910 no se usan en esta etapa'


def p0(mes):
    """Período consecutivo: el mes anterior es siempre p0 − 1."""
    return (mes // 100) * 12 + mes % 100


H_MAX = max([2] + VARIANTES[VARIANTE]['lags'] + [VARIANTES[VARIANTE]['tendencia'] or 0])
if p0(min(MESES_TRAIN)) - H_MAX < p0(201901):
    print(f'Aviso: con {VARIANTE}, {min(MESES_TRAIN)} no tiene {H_MAX} meses previos en el dataset')
print('Configuración:', CONFIG, '· variante:', VARIANTE)

TIEMPOS = []


def memoria_gb():
    """Memoria máxima del proceso hasta ahora (macOS la da en bytes; Linux, en KB)."""
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(r / (1e9 if sys.platform == 'darwin' else 1e6), 2)


def registrar(paso, t0):
    segundos = time.time() - t0
    TIEMPOS.append({'config': CONFIG, 'variante': VARIANTE, 'paso': paso,
                    'segundos': round(segundos, 1), 'memoria_max_gb': memoria_gb()})
    print(f'{paso}: {segundos:.0f} s · memoria máxima {memoria_gb()} GB')

Configuración: train201905-201906_rotosNA_nominal · variante: LT4


## 2. Valores esperados

Salen de la etapa 1 (`01_clases_por_mes.csv`) y de consultas de solo lectura del 2026-11-02. El notebook se **frena** si algo no coincide.

La cantidad de variables por variante sale de la clasificación (sección 5): A = 152 originales + 4 × 152 (lag y delta 1 y 2).

In [2]:
ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116,
    201905: 126_763, 201906: 128_370, 201907: 129_853, 201908: 131_775,
}
ESPERADO_CLASES = {   # BAJA+1, BAJA+2
    201901: (647, 697), 201902: (700, 696), 201903: (700, 774), 201904: (776, 575),
    201905: (576, 658), 201906: (665, 589), 201907: (595, 656), 201908: (659, 543),
}
ESPERADO_N_VARIABLES = {'A': 760, 'L3': 1_044, 'L4': 1_328, 'T3': 1_284, 'T4': 1_284, 'LT3': 1_568, 'LT4': 1_852}
ESPERADO_COBERTURA_LAG4 = {201905: 95.5, 201906: 95.3, 201908: 93.7}   # = h4 de 01_cobertura_historia.csv
VARIABLE_COBERTURA = 'cproductos'   # conteo sin nulos ni meses rotos en 201901–201908
ESPERADO_ROTOS = 23

## 3. Semillas

Las mismas de la competencia 1: todos los primos de 6 dígitos, y se sortean 20 sin reemplazo con `default_rng(979201)`. Salen siempre las mismas y en el mismo orden. El TP usa las primeras 10; esta etapa, las primeras `N_SEMILLAS`.

In [3]:
def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


SEMILLAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()
SEMILLAS_A_USAR = SEMILLAS[:N_SEMILLAS]
print('Semillas:', SEMILLAS_A_USAR)

Semillas: [294103, 342929, 161093, 677561, 431077]


## 4. Carga del dataset base

Se cargan **solo los meses de 201901 a `MES_VALID`**: los anteriores al entrenamiento hacen falta para la historia; 201909 en adelante no se cargan. Orden por mes y cliente, como el parquet.

In [4]:
t0 = time.time()
con = duckdb.connect()
base = con.execute(f"""
    select * from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes <= {MES_VALID}
    order by foto_mes, numero_de_cliente
""").df()
con.close()

por_mes = base['foto_mes'].value_counts().sort_index().to_dict()
assert por_mes == ESPERADO_FILAS_POR_MES, por_mes
assert base['foto_mes'].max() == MES_VALID < 201909

clases = base.groupby('foto_mes')['clase_ternaria'].value_counts().unstack()
for mes, (b1, b2) in ESPERADO_CLASES.items():
    assert (clases.loc[mes, 'BAJA+1'], clases.loc[mes, 'BAJA+2']) == (b1, b2), mes
assert base['clase_ternaria'].notna().all()
registrar('carga', t0)
print(f'{len(base):,} filas · {base.shape[1]} columnas')

carga: 1 s · memoria máxima 2.91 GB
1,017,435 filas · 155 columnas


## 5. Clasificación de las variables por tipo

Por el **significado** (diccionario de datos) y por los **valores** que toma cada variable. **Nunca mirando la clase.**

| Tipo | Historia que agregan las variantes |
|---|---|
| Montos y conteos (129) | lag y delta; pendiente, promedio, mínimo y máximo |
| Banderas (8) | lag y delta (el "cambio": −1, 0 o 1); permanencia (meses activa en la ventana) |
| Estados (5) | lag y "cambió" (0 o 1) |
| Fechas (8) y edad/antigüedad (2) | nada: son días o meses que avanzan solos |

El control A no usa la clasificación: lleva lag y delta 1 y 2 de **las 152**, como la consigna.

Dos diferencias con el diccionario, por los valores de 2019: `ccajas_transacciones` solo vale 0 o 1 (bandera) y `ccaja_seguridad` vale de 0 a 6 (conteo).

In [5]:
CLAVES = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
VARIABLES = [c for c in base.columns if c not in CLAVES]

BANDERAS = ['active_quarter', 'cliente_vip', 'internet', 'cdescubierto_preacordado',
            'tcallcenter', 'thomebanking', 'tmobile_app', 'ccajas_transacciones']
ESTADOS = ['tcuentas', 'Visa_status', 'Master_status', 'Visa_delinquency', 'Master_delinquency']
FECHAS = [f'{t}_{c}' for t in ('Visa', 'Master') for c in ('Fvencimiento', 'Finiciomora', 'fultimo_cierre', 'fechaalta')]
EDADES = ['cliente_edad', 'cliente_antiguedad']
MONTOS = [v for v in VARIABLES if v not in BANDERAS + ESTADOS + FECHAS + EDADES]
EN_PESOS = [v for v in MONTOS if v.startswith(('m', 'Visa_m', 'Master_m'))]   # para HISTORIA_SOBRE = 'ranking'

TIPO = {**{v: 'monto o conteo' for v in MONTOS}, **{v: 'bandera' for v in BANDERAS},
        **{v: 'estado' for v in ESTADOS}, **{v: 'fecha' for v in FECHAS}, **{v: 'edad' for v in EDADES}}

assert len(VARIABLES) == 152 and set(BANDERAS + ESTADOS + FECHAS + EDADES) <= set(VARIABLES)
assert (len(MONTOS), len(BANDERAS), len(ESTADOS), len(FECHAS), len(EDADES)) == (129, 8, 5, 8, 2)
for v in BANDERAS:
    assert set(base[v].dropna().unique()) <= {0, 1}, f'{v} no es 0/1'
for v in ESTADOS:
    assert base[v].nunique() <= 4, f'{v} tiene demasiados valores para ser un estado'
print(f'{len(MONTOS)} montos y conteos ({len(EN_PESOS)} en pesos) · {len(BANDERAS)} banderas · '
      f'{len(ESTADOS)} estados · {len(FECHAS)} fechas · {len(EDADES)} edades')

129 montos y conteos (73 en pesos) · 8 banderas · 5 estados · 8 fechas · 2 edades


## 6. Meses rotos (decisión 2)

Con `TRATAR_ROTOS = True`, cada caso (variable, mes) de `01_meses_rotos.csv` pasa a **NA en todo el mes**, **antes** de calcular la historia. Así un 0 falso no se convierte en un "cayó a cero" en el delta del mes siguiente. Se aplica igual al control y a la variante.

Solo se aplican los casos de los meses cargados. Los de 201910 se aplican cuando ese mes entre, en la evaluación final.

In [6]:
rotos = pd.read_csv(ROTOS_CSV)
assert len(rotos) == ESPERADO_ROTOS
rotos_cargados = rotos[rotos['foto_mes'].isin(por_mes)]

if TRATAR_ROTOS:
    for v, mes in rotos_cargados[['variable', 'foto_mes']].itertuples(index=False):
        base.loc[base['foto_mes'] == mes, v] = np.nan
    print(f'{len(rotos_cargados)} casos a NA:')
    print(rotos_cargados.groupby('foto_mes')['variable'].apply(list).to_string())
else:
    print('Meses rotos sin tratar')

11 casos a NA:
foto_mes
201904    [ctarjeta_visa_debitos_automaticos, mttarjeta_...
201905    [ccomisiones_otras, mactivos_margen, mcomision...
201907         [Visa_fultimo_cierre, Master_fultimo_cierre]


## 7. Índices de historia

Para cada fila de los meses que se usan (`MESES_TRAIN` y `MES_VALID`) se busca **la fila del mismo cliente k meses antes, por calendario** (`p0 − k`). Si el cliente no estaba ese mes, el índice es −1 y la historia queda en NA.

Por calendario y por fila solo difieren en 53 filas de 2019: los clientes que se van y vuelven.

In [7]:
t0 = time.time()
periodos = (base['foto_mes'] // 100 * 12 + base['foto_mes'] % 100).to_numpy(np.int64)
clave = base['numero_de_cliente'].to_numpy(np.int64) * 100_000 + periodos
indice = pd.Index(clave)
assert indice.is_unique

meses_objetivo = MESES_TRAIN + [MES_VALID]
filas = np.flatnonzero(base['foto_mes'].isin(meses_objetivo).to_numpy())
mes_fila = base['foto_mes'].to_numpy()[filas]
n_train = int(np.isin(mes_fila, MESES_TRAIN).sum())
assert np.isin(mes_fila[:n_train], MESES_TRAIN).all() and (mes_fila[n_train:] == MES_VALID).all()

POS = {k: indice.get_indexer(clave[filas] - k) for k in range(H_MAX + 1)}
assert (POS[0] == filas).all()
print(f'{len(filas):,} filas ({n_train:,} de entrenamiento) · historia hasta k = {H_MAX}')

386,908 filas (255,133 de entrenamiento) · historia hasta k = 4


## 8. Variables históricas

Se arma la tabla de la variante, y **A es un subconjunto de sus columnas**: las primeras 760. Así A y la variante comparten exactamente las mismas filas, en el mismo orden, y las mismas columnas de base.

| Bloque | Columnas |
|---|---|
| base | las 152 del mes |
| A | `_lag1`, `_delta1`, `_lag2`, `_delta2` de las 152 |
| L3, L4 | `_lag3`, `_delta3` (`_cambio3` en los estados), y lo mismo con 4 |
| T3, T4 | `_pend`, `_prom`, `_min`, `_max` (montos) y `_perm` (banderas), con ventana de m − h a m |

- **Delta:** valor del mes menos el lag.
- **Pendiente:** mínimos cuadrados contra el mes, en unidades por mes, con los meses presentes de la ventana. Con menos de 2 puntos queda en NA.
- **Ranking** (solo con `HISTORIA_SOBRE = 'ranking'`): la historia de los montos en pesos se calcula sobre el percentil dentro del mes. El valor del mes queda en pesos.

La tabla se arma en memoria en cada corrida (no se guarda): son unos 2,9 GB para LT4.

In [8]:
t0 = time.time()
EXTRA = VARIANTES[VARIANTE]
h = EXTRA['tendencia']

especificacion = [('original', v, 0) for v in VARIABLES]
especificacion += [(op, v, k) for k in (1, 2) for op in ('lag', 'delta') for v in VARIABLES]
for k in EXTRA['lags']:
    especificacion += [(op, v, k) for op in ('lag', 'delta') for v in MONTOS + BANDERAS]
    especificacion += [(op, v, k) for op in ('lag', 'cambio') for v in ESTADOS]
if h:
    especificacion += [(op, v, h) for op in ('pend', 'prom', 'min', 'max') for v in MONTOS]
    especificacion += [('perm', v, h) for v in BANDERAS]


def nombre(op, v, k):
    return v if op == 'original' else f'{v}_{op}{k}'


def bloque(op, k):
    if op == 'original':
        return 'base'
    if op in ('lag', 'delta', 'cambio'):
        return 'A' if k <= 2 else f'L{k}'
    return f'T{k}'


COLUMNAS = [nombre(*e) for e in especificacion]
BLOQUES = [bloque(op, k) for op, _, k in especificacion]


def columnas_de(variante):
    """Slice de columnas de la variante. A siempre es el comienzo de la tabla."""
    ex = VARIANTES[variante]
    permitidos = {'base', 'A'} | {f'L{k}' for k in ex['lags']} | ({f'T{ex["tendencia"]}'} if ex['tendencia'] else set())
    dentro = [b in permitidos for b in BLOQUES]
    n = sum(dentro)
    assert all(dentro[:n]) and not any(dentro[n:]), 'Las columnas de la variante tienen que ser un prefijo'
    return slice(0, n)


# Historia: X es la escala de la historia (pesos o percentil dentro del mes)
X_mes = base[VARIABLES].to_numpy(np.float32)
X = X_mes
if HISTORIA_SOBRE == 'ranking':
    X = X_mes.copy()
    rangos = base.groupby('foto_mes')[EN_PESOS].rank(pct=True).to_numpy(np.float32)
    X[:, [VARIABLES.index(v) for v in EN_PESOS]] = rangos
    del rangos


def historia(k):
    out = X[POS[k]]
    out[POS[k] < 0] = np.nan
    return out


H = {k: historia(k) for k in range(H_MAX + 1)}
col = {v: j for j, v in enumerate(VARIABLES)}
F = np.empty((len(filas), len(especificacion)), dtype=np.float32)
pendientes = {}
for i, (op, v, k) in enumerate(especificacion):
    j = col[v]
    if op == 'original':
        F[:, i] = X_mes[filas, j]
    elif op == 'lag':
        F[:, i] = H[k][:, j]
    elif op == 'delta':
        F[:, i] = H[0][:, j] - H[k][:, j]
    elif op == 'cambio':
        a, b = H[0][:, j], H[k][:, j]
        F[:, i] = np.where(np.isnan(a) | np.isnan(b), np.nan, (a != b).astype(np.float32))
    else:
        pendientes[(op, v)] = i

if h:
    t = -np.arange(h + 1, dtype=np.float64)[:, None]          # mes m = 0, m − 1 = −1, …
    with np.errstate(invalid='ignore', divide='ignore'):
        for v in MONTOS + BANDERAS:
            S = np.stack([H[k][:, col[v]] for k in range(h + 1)]).astype(np.float64)
            valido = ~np.isnan(S)
            cuantos = valido.sum(axis=0)
            S0 = np.where(valido, S, 0.0)
            if v in BANDERAS:
                F[:, pendientes[('perm', v)]] = np.where(cuantos > 0, S0.sum(axis=0), np.nan)
                continue
            prom = S0.sum(axis=0) / cuantos
            dt = np.where(valido, t - (valido * t).sum(axis=0) / cuantos, 0.0)
            pend = (dt * (S0 - prom)).sum(axis=0) / (dt ** 2).sum(axis=0)
            pend[cuantos < 2] = np.nan
            F[:, pendientes[('pend', v)]] = pend
            F[:, pendientes[('prom', v)]] = prom
            F[:, pendientes[('min', v)]] = np.where(cuantos > 0, np.where(valido, S, np.inf).min(axis=0), np.nan)
            F[:, pendientes[('max', v)]] = np.where(cuantos > 0, np.where(valido, S, -np.inf).max(axis=0), np.nan)

del H, X, X_mes
COLS_A, COLS_V = columnas_de('A'), columnas_de(VARIANTE)
pd.DataFrame({'columna': COLUMNAS, 'variable': [v for _, v, _ in especificacion],
              'tipo': [TIPO[v] for _, v, _ in especificacion], 'operacion': [op for op, _, _ in especificacion],
              'k_o_h': [k for _, _, k in especificacion], 'bloque': BLOQUES,
              'en_A': [i < COLS_A.stop for i in range(len(COLUMNAS))],
              f'en_{VARIANTE}': True}).to_csv(VARIABLES_CSV, index=False)
registrar('variables', t0)
print(f'Tabla: {F.shape[0]:,} filas × {F.shape[1]:,} columnas · {F.nbytes / 1e9:.1f} GB · '
      f'A = {COLS_A.stop} columnas · {VARIANTE} = {COLS_V.stop}')

variables: 10 s · memoria máxima 5.19 GB
Tabla: 386,908 filas × 1,852 columnas · 2.9 GB · A = 760 columnas · LT4 = 1852


## 9. Controles de las variables

1. Cantidad de columnas de A y de la variante.
2. **Lags contra la cobertura:** el % de lag k no nulo de `cproductos` (un conteo sin nulos ni meses rotos en 2019) es el % de clientes que estaban k meses antes, igual que `01_cobertura_historia.csv`.
3. **Deltas:** `cliente_antiguedad_delta1` vale 1 en todas las filas con lag 1.
4. **Lags por otro camino:** en 2.000 filas al azar, el lag de 5 variables buscado con un `merge` de pandas coincide con la tabla.
5. **Tendencias a mano:** en 3 clientes con la ventana completa, `np.polyfit`, promedio, mínimo y máximo coinciden.
6. **Meses rotos:** NA en su mes y en los lags que lo miran.

In [9]:
nombre_a_col = {c: i for i, c in enumerate(COLUMNAS)}
assert COLS_A.stop == ESPERADO_N_VARIABLES['A'] and COLS_V.stop == ESPERADO_N_VARIABLES[VARIANTE]

# 2. Lags contra la cobertura
cobertura = pd.read_csv(COBERTURA_CSV, index_col='foto_mes')
lags = sorted({k for op, _, k in especificacion if op == 'lag'})
for k in [k for k in lags if k >= 2]:
    presente = ~np.isnan(F[:, nombre_a_col[f'{VARIABLE_COBERTURA}_lag{k}']])
    pct = pd.Series(presente).groupby(mes_fila).mean().mul(100)
    for mes, valor in pct.items():
        assert abs(valor - cobertura.loc[mes, f'h{k}']) <= 0.051, (k, mes, valor)
if 4 in lags:
    lag4 = pd.Series(~np.isnan(F[:, nombre_a_col[f'{VARIABLE_COBERTURA}_lag4']])).groupby(mes_fila).mean().mul(100).round(1)
    assert {m: lag4[m] for m in ESPERADO_COBERTURA_LAG4 if m in lag4} == \
           {m: v for m, v in ESPERADO_COBERTURA_LAG4.items() if m in lag4}

# 3. Deltas
d1, l1 = F[:, nombre_a_col['cliente_antiguedad_delta1']], F[:, nombre_a_col['cliente_antiguedad_lag1']]
assert (d1[~np.isnan(l1)] == 1).all()

# 4. Lags por otro camino (variables fuera de EN_PESOS: valen con 'nominal' y con 'ranking')
rng = np.random.default_rng(SEMILLA_PRIMIGENIA)
muestra = rng.choice(len(filas), size=2_000, replace=False)
aux = base[['numero_de_cliente', 'foto_mes']].assign(p=periodos)
VARIABLES_CONTROL = ['ctrx_quarter', 'thomebanking', 'Visa_status', 'cpayroll_trx', 'ccomisiones_otras']
for k in lags:
    buscado = aux.iloc[filas[muestra]].assign(p=lambda d: d['p'] - k).merge(
        base[['numero_de_cliente'] + VARIABLES_CONTROL].assign(p=periodos),
        on=['numero_de_cliente', 'p'], how='left')
    for v in VARIABLES_CONTROL:
        np.testing.assert_array_equal(F[muestra, nombre_a_col[f'{v}_lag{k}']], buscado[v].to_numpy(np.float32))

# 5. Tendencias a mano
if h:
    v = 'ctrx_quarter'
    valores = pd.Series(base[v].to_numpy(np.float64), index=indice)
    ventana = np.stack([valores.reindex(clave[filas[muestra]] - k).to_numpy() for k in range(h + 1)])
    completas = np.flatnonzero(~np.isnan(ventana).any(axis=0))
    for j in rng.choice(completas, size=3, replace=False):
        i, y = muestra[j], ventana[:, j]
        tt = -np.arange(h + 1)
        np.testing.assert_allclose(
            [F[i, nombre_a_col[f'{v}_pend{h}']], F[i, nombre_a_col[f'{v}_prom{h}']],
             F[i, nombre_a_col[f'{v}_min{h}']], F[i, nombre_a_col[f'{v}_max{h}']]],
            [np.polyfit(tt, y, 1)[0], y.mean(), y.min(), y.max()], rtol=1e-4, atol=1e-4)

# 6. Meses rotos: NA en su mes y en los lags que lo miran
if TRATAR_ROTOS:
    revisados = 0
    for v, mes in rotos_cargados[['variable', 'foto_mes']].itertuples(index=False):
        for k in [0] + lags:
            destino = p0(mes) + k
            en_destino = (mes_fila // 100 * 12 + mes_fila % 100) == destino
            columna = v if k == 0 else f'{v}_lag{k}'
            if en_destino.any() and columna in nombre_a_col:
                c = nombre_a_col[columna]
                assert np.isnan(F[en_destino, c]).all(), (v, mes, k)
                revisados += 1
    print(f'Meses rotos: {revisados} combinaciones (variable, mes, lag) en NA')
print('Controles de variables OK')

Meses rotos: 29 combinaciones (variable, mes, lag) en NA
Controles de variables OK


## 10. Optuna sobre A

- **Busca** los 5 hiperparámetros de la consigna, con los rangos de la competencia 1 (`num_leaves`, `min_data_in_leaf`, `learning_rate`, `feature_fraction` y `bagging_fraction`).
- **Fijos:** `gbdt`, `bagging_freq` 1, sin regularizaciones, `max_bin` 31 (como en la competencia 1), `deterministic`.
- **Cantidad de árboles:** la da el *early stopping* sobre la AUC de `MES_VALID` (contra `BAJA+2`), hasta `ARBOLES_MAX`.
- Una semilla de LightGBM (la primera); el sampler TPE usa la semilla primigenia.
- El estudio se guarda en SQLite: si se corta, se retoma.

Los mejores hiperparámetros de A se usan **también para la variante** (decisión 8).

In [10]:
FIJOS = {'objective': 'binary', 'metric': 'auc', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False,
         'num_threads': N_HILOS, 'deterministic': True, 'force_col_wise': True, 'verbose': -1}

clase = base['clase_ternaria'].to_numpy()[filas]
y_train = np.isin(clase[:n_train], POSITIVOS).astype(np.int8)
y_auc = np.isin(clase[n_train:], POSITIVOS_AUC).astype(np.int8)
es_baja2 = clase[n_train:] == 'BAJA+2'
print(f'Entrenamiento: {n_train:,} filas, {y_train.sum():,} positivas · validación {MES_VALID}: '
      f'{len(y_auc):,} filas, {es_baja2.sum()} BAJA+2')


def datasets(cols):
    dtrain = lgb.Dataset(F[:n_train, cols], label=y_train, feature_name=COLUMNAS[cols],
                         params={'max_bin': 31, 'feature_pre_filter': False}, free_raw_data=False)
    dvalid = lgb.Dataset(F[n_train:, cols], label=y_auc, reference=dtrain, free_raw_data=False)
    return dtrain, dvalid


def buscar(nombre_estudio, cols, n_pruebas):
    dtrain, dvalid = datasets(cols)

    def objetivo(trial):
        p = {**FIJOS,
             'num_leaves': trial.suggest_int('num_leaves', 8, 512, log=True),
             'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 2000, log=True),
             'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
             'feature_fraction': trial.suggest_float('feature_fraction', 0.2, 1.0),
             'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
             'seed': SEMILLAS[0]}
        t0 = time.time()
        modelo = lgb.train(p, dtrain, num_boost_round=ARBOLES_MAX, valid_sets=[dvalid], valid_names=['valid'],
                           callbacks=[lgb.early_stopping(PACIENCIA, verbose=False)])
        auc = modelo.best_score['valid']['auc']
        trial.set_user_attr('arboles', modelo.best_iteration)
        trial.set_user_attr('segundos', round(time.time() - t0, 1))
        print(f'prueba {trial.number}: AUC {auc:.5f} · {modelo.best_iteration} árboles · hojas {p["num_leaves"]} · '
              f'mdl {p["min_data_in_leaf"]} · lr {p["learning_rate"]:.3f} · ff {p["feature_fraction"]:.2f} · '
              f'bf {p["bagging_fraction"]:.2f} · {time.time() - t0:.0f} s')
        return auc

    estudio = optuna.create_study(study_name=nombre_estudio, storage=STORAGE, direction='maximize',
                                  load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=SEMILLA_PRIMIGENIA))
    hechas = sum(t.state == optuna.trial.TrialState.COMPLETE for t in estudio.trials)
    estudio.optimize(objetivo, n_trials=max(0, n_pruebas - hechas))
    return estudio, dtrain


t0 = time.time()
estudio_A, dtrain_A = buscar('02_minimo_A', COLS_A, N_PRUEBAS_OPTUNA)
completas = [t for t in estudio_A.trials if t.state == optuna.trial.TrialState.COMPLETE]
segundos_prueba = float(np.mean([t.user_attrs['segundos'] for t in completas]))
registrar('optuna A', t0)
TIEMPOS.append({'config': CONFIG, 'variante': 'A', 'paso': 'una prueba de Optuna (media)',
                'segundos': round(segundos_prueba, 1), 'memoria_max_gb': memoria_gb()})

MEJORES = {**FIJOS, **estudio_A.best_trial.params}
ARBOLES = estudio_A.best_trial.user_attrs['arboles']
print(f'Mejor prueba: {estudio_A.best_trial.number} · AUC {estudio_A.best_value:.5f} · {ARBOLES} árboles')
print({k: round(v, 4) if isinstance(v, float) else v for k, v in estudio_A.best_trial.params.items()})

Entrenamiento: 255,133 filas, 2,488 positivas · validación 201908: 131,775 filas, 543 BAJA+2
prueba 0: AUC 0.90607 · 724 árboles · hojas 71 · mdl 11 · lr 0.011 · ff 0.38 · bf 0.68 · 48 s
prueba 1: AUC 0.90787 · 328 árboles · hojas 87 · mdl 468 · lr 0.013 · ff 0.76 · bf 0.83 · 34 s
prueba 2: AUC 0.90705 · 415 árboles · hojas 275 · mdl 1230 · lr 0.013 · ff 0.44 · bf 0.75 · 46 s
prueba 3: AUC 0.90343 · 43 árboles · hojas 66 · mdl 676 · lr 0.079 · ff 0.37 · bf 0.74 · 9 s
prueba 4: AUC 0.90563 · 700 árboles · hojas 19 · mdl 20 · lr 0.013 · ff 0.37 · bf 0.47 · 30 s
prueba 5: AUC 0.89726 · 55 árboles · hojas 243 · mdl 201 · lr 0.056 · ff 0.48 · bf 0.45 · 17 s
prueba 6: AUC 0.89907 · 45 árboles · hojas 342 · mdl 573 · lr 0.099 · ff 0.77 · bf 0.74 · 25 s
prueba 7: AUC 0.90154 · 149 árboles · hojas 340 · mdl 86 · lr 0.046 · ff 0.23 · bf 0.92 · 29 s
prueba 8: AUC 0.90210 · 126 árboles · hojas 41 · mdl 15 · lr 0.035 · ff 0.65 · bf 0.56 · 10 s
prueba 9: AUC 0.89415 · 143 árboles · hojas 37 · mdl 22

## 11. (Opcional) Optuna propio de la variante

Solo con `OPTUNA_VARIANTE = True`. Sirve para ver si la variante mejora mucho con su propia búsqueda. **No cambia** los modelos finales, que usan los hiperparámetros de A.

In [11]:
if OPTUNA_VARIANTE:
    t0 = time.time()
    estudio_V, _ = buscar(f'02_minimo_{VARIANTE}', COLS_V, N_PRUEBAS_VARIANTE)
    registrar(f'optuna {VARIANTE}', t0)
    print(f'Mejor AUC propia de {VARIANTE}: {estudio_V.best_value:.5f} · la de A: {estudio_A.best_value:.5f}')
else:
    print('Se saltea (OPTUNA_VARIANTE = False)')

Se saltea (OPTUNA_VARIANTE = False)


## 12. Modelos finales: A y la variante, con las mismas semillas

Mismos hiperparámetros y misma cantidad de árboles (los de la mejor prueba de A), mismas filas y mismas semillas. Para cada (variante, semilla):
- AUC en `MES_VALID` contra `BAJA+2`;
- **curva de ganancia**, ordenando por probabilidad;
- **meseta:** el máximo de la curva suavizada (media móvil centrada de 501), y su N.

Cada resultado se agrega a `resultados/02_minimo_semillas.csv` apenas termina: si la corrida se corta, las semillas hechas no se repiten.

**Control de reproducibilidad:** A con la primera semilla tiene que dar la misma AUC que la mejor prueba de Optuna.

In [12]:
def curva_ganancia(prob, es_b2, largo=LARGO_CURVA):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:largo].astype(np.float64)


def meseta(curva):
    """(N, ganancia) en el máximo de la curva suavizada."""
    s = pd.Series(curva).rolling(VENTANA_SUAVIZADO, center=True, min_periods=VENTANA_SUAVIZADO).mean().to_numpy()
    i = int(np.nanargmax(s))
    return i + 1, float(s[i])


TOPE_GANANCIA = es_baja2.sum() * GANANCIA_ACIERTO
t_modelos = time.time()
hechos = pd.read_csv(SEMILLAS_CSV) if SEMILLAS_CSV.exists() else pd.DataFrame(columns=['config', 'variante', 'semilla'])

for variante, cols in [('A', COLS_A), (VARIANTE, COLS_V)]:
    dtrain = dtrain_A if variante == 'A' else datasets(cols)[0]
    for semilla in SEMILLAS_A_USAR:
        if ((hechos['config'] == CONFIG) & (hechos['variante'] == variante) & (hechos['semilla'] == semilla)).any():
            print(f'{variante} · semilla {semilla}: ya estaba')
            continue
        t0 = time.time()
        modelo = lgb.train({**MEJORES, 'seed': semilla}, dtrain, num_boost_round=ARBOLES)
        prob = modelo.predict(F[n_train:, cols], num_threads=N_HILOS)
        auc = roc_auc_score(y_auc, prob)
        curva = curva_ganancia(prob, es_baja2)
        n, g = meseta(curva)
        np.save(CURVAS / f'{variante}_{semilla}.npy', curva)
        segundos = round(time.time() - t0, 1)

        assert 0.8 < auc < 0.98, f'AUC {auc:.4f} fuera de lo razonable: buscar una fuga o un error antes de seguir'
        assert curva.max() <= TOPE_GANANCIA
        if variante == 'A' and semilla == SEMILLAS[0]:
            assert abs(auc - estudio_A.best_value) < 1e-6, (auc, estudio_A.best_value)

        fila = pd.DataFrame([{'config': CONFIG, 'variante': variante, 'semilla': semilla,
                              'n_columnas': cols.stop, 'arboles': ARBOLES, f'auc_{MES_VALID}': round(auc, 6),
                              'ganancia_meseta_M': round(g / 1e6, 3), 'N_meseta': n, 'segundos': segundos}])
        fila.to_csv(SEMILLAS_CSV, mode='a', header=not SEMILLAS_CSV.exists(), index=False)
        print(f'{variante} · semilla {semilla}: AUC {auc:.5f} · ganancia {g / 1e6:,.1f} M (N {n:,}) · {segundos:.0f} s')
    del dtrain

registrar('modelos finales', t_modelos)
print(f'Tope teórico de ganancia en {MES_VALID}: {TOPE_GANANCIA / 1e6:,.1f} M ({es_baja2.sum()} BAJA+2)')

A · semilla 294103: AUC 0.90928 · ganancia 195.4 M (N 4,691) · 39 s
A · semilla 342929: AUC 0.90712 · ganancia 195.7 M (N 4,437) · 40 s
A · semilla 161093: AUC 0.90646 · ganancia 196.1 M (N 3,908) · 50 s
A · semilla 677561: AUC 0.90790 · ganancia 194.1 M (N 5,109) · 53 s
A · semilla 431077: AUC 0.90745 · ganancia 193.8 M (N 4,146) · 44 s
LT4 · semilla 294103: AUC 0.90850 · ganancia 195.9 M (N 3,731) · 112 s
LT4 · semilla 342929: AUC 0.90871 · ganancia 198.6 M (N 5,412) · 112 s
LT4 · semilla 161093: AUC 0.90991 · ganancia 201.2 M (N 4,343) · 117 s
LT4 · semilla 677561: AUC 0.90942 · ganancia 201.5 M (N 4,872) · 99 s
LT4 · semilla 431077: AUC 0.90850 · ganancia 195.5 M (N 4,235) · 92 s
modelos finales: 758 s · memoria máxima 5.22 GB
Tope teórico de ganancia en 201908: 582.4 M (543 BAJA+2)


## 13. Resumen (solo descriptivo)

Tabla por semilla y diferencia pareada (variante − A). **No es una conclusión sobre la hipótesis:**
- la ganancia se mide en el mismo mes que eligió los hiperparámetros;
- son pocas semillas;
- no hay prueba estadística.

Sirve para ver que todo funciona y si hay alguna señal.

In [13]:
res = pd.read_csv(SEMILLAS_CSV)
res = res[(res['config'] == CONFIG) & res['semilla'].isin(SEMILLAS_A_USAR) & res['variante'].isin(['A', VARIANTE])]
ganancia = res.pivot(index='semilla', columns='variante', values='ganancia_meseta_M')[['A', VARIANTE]]
auc = res.pivot(index='semilla', columns='variante', values=f'auc_{MES_VALID}')[['A', VARIANTE]]
ganancia['diferencia'] = ganancia[VARIANTE] - ganancia['A']
auc['diferencia'] = auc[VARIANTE] - auc['A']

print(f'Ganancia en la meseta, {MES_VALID} (millones de $):')
print(ganancia.round(2).to_string())
print(ganancia.agg(['mean', 'std']).round(2).to_string())
print(f'\nAUC en {MES_VALID}:')
print(auc.round(5).to_string())
print(f'\n{VARIANTE} le gana a A en {(ganancia["diferencia"] > 0).sum()} de {len(ganancia)} semillas (descriptivo)')

Ganancia en la meseta, 201908 (millones de $):
variante       A     LT4  diferencia
semilla                             
161093    196.11  201.20        5.10
294103    195.37  195.87        0.50
342929    195.67  198.64        2.97
431077    193.78  195.48        1.70
677561    194.13  201.48        7.36
variante       A     LT4  diferencia
mean      195.01  198.54        3.53
std         1.01    2.84        2.74

AUC en 201908:
variante        A      LT4  diferencia
semilla                               
161093    0.90646  0.90990     0.00345
294103    0.90928  0.90850    -0.00078
342929    0.90712  0.90871     0.00159
431077    0.90745  0.90850     0.00105
677561    0.90790  0.90942     0.00152

LT4 le gana a A en 5 de 5 semillas (descriptivo)


## 14. Tiempos, memoria y proyección

Se guardan en `resultados/02_tiempos.csv`. La proyección es una **cota superior** del diseño completo en esta máquina: supone que todas las variantes cuestan como la variante de este experimento, que es la más grande. Los valores salen de `DISENO`, en los parámetros.

In [14]:
seg_semilla = res.groupby('variante')['segundos'].mean()
for variante, s in seg_semilla.items():
    TIEMPOS.append({'config': CONFIG, 'variante': variante, 'paso': 'un modelo final (media)',
                    'segundos': round(s, 1), 'memoria_max_gb': memoria_gb()})
tiempos = pd.DataFrame(TIEMPOS).assign(fecha_hora=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
                                       maquina=platform.node(), hilos=N_HILOS)
tiempos.to_csv(TIEMPOS_CSV, mode='a', header=not TIEMPOS_CSV.exists(), index=False)
print(tiempos[['variante', 'paso', 'segundos', 'memoria_max_gb']].to_string(index=False))

modelos = DISENO['variantes'] * DISENO['semillas'] * DISENO['meses_evaluacion'] * seg_semilla[VARIANTE]
busqueda = DISENO['pruebas_optuna'] * segundos_prueba
print(f'\nProyección del diseño completo (cota superior, esta máquina): '
      f'{(modelos + busqueda) / 3600:.1f} h = {modelos / 3600:.1f} h de modelos + {busqueda / 3600:.1f} h de Optuna')

variante                         paso  segundos  memoria_max_gb
     LT4                        carga       0.7            2.91
     LT4                    variables      10.2            5.19
     LT4                     optuna A     687.4            5.22
       A una prueba de Optuna (media)      34.3            5.22
     LT4              modelos finales     758.0            5.22
       A      un modelo final (media)      45.3            5.22
     LT4      un modelo final (media)     106.3            5.22

Proyección del diseño completo (cota superior, esta máquina): 4.6 h = 4.1 h de modelos + 0.5 h de Optuna


## 15. Cierre

- **Verificación de la ficha:** los controles 1 a 8 y 11 se cumplen si el notebook llegó hasta acá sin frenarse. El 10 (retomar) se prueba cortando el kernel en la sección 10 y volviendo a correr. El 12 son los tiempos de arriba.
- **Registro:** anotar en `decisiones.md` el resultado y llevar los tiempos a §3.2 del `CLAUDE.md`.
- **Lo que sigue:** el primer punto de replanificación (etapa 3), con estos tiempos y la respuesta de los profes.